#### Import Libraries and General Settings

In [29]:
import sys
import pandas as pd
import optuna
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
import optuna.logging

optuna.logging.set_verbosity(optuna.logging.WARNING)
optuna.logging.set_verbosity(optuna.logging.ERROR)

import xgboost, sklearn

print("Python:", sys.version)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("Optuna:", optuna.__version__)
print("XGBoost:", xgboost.__version__)
print("Sklearn:", sklearn.__version__)

Python: 3.11.14 (main, Oct 10 2025, 08:54:03) [GCC 11.4.0]
numpy: 2.4.2
pandas: 3.0.1
Optuna: 4.7.0
XGBoost: 3.2.0
Sklearn: 1.4.2


#### Load data and name columns

In [30]:
df = pd.read_csv('dataset/crx.data', header=None, 
                 names = ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 
                          'A9', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'Target'])
df

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15,Target
0,b,30.83,0.0000,u,g,w,v,1.2500,t,t,1,f,g,00202,0,+
1,a,58.67,4.4600,u,g,q,h,3.0400,t,t,6,f,g,00043,560,+
2,a,24.50,0.5000,u,g,q,h,1.5000,t,f,0,f,g,00280,824,+
3,b,27.83,1.5400,u,g,w,v,3.7500,t,t,5,t,g,00100,3,+
4,b,20.17,5.6250,u,g,w,v,1.7100,t,f,0,f,s,00120,0,+
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
685,b,21.08,10.0850,y,p,e,h,1.2500,f,f,0,f,g,00260,0,-
686,a,22.67,0.7500,u,g,c,v,2.0000,f,t,2,t,g,00200,394,-
687,a,25.25,13.5000,y,p,ff,ff,2.0000,f,t,1,t,g,00200,1,-
688,b,17.92,0.2050,u,g,aa,v,0.0400,f,f,0,f,g,00280,750,-


#### Statistical summary of the columns

In [31]:
df.describe()

,A3,A8,A11,A15
count,690.0000,690.0000,690.0000,690.0000
mean,4.7587,2.2234,2.4000,"1,017.3855"
std,4.9782,3.3465,4.8629,"5,210.1026"
min,0.0000,0.0000,0.0000,0.0000
25%,1.0000,0.1650,0.0000,0.0000
50%,2.7500,1.0000,0.0000,5.0000
75%,7.2075,2.6250,3.0000,395.5000
max,28.0000,28.5000,67.0000,"100,000.0000"


#### Number of rows, columns, data types, number of non-null values

In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 690 entries, 0 to 689
Data columns (total 16 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   A1      690 non-null    str    
 1   A2      690 non-null    str    
 2   A3      690 non-null    float64
 3   A4      690 non-null    str    
 4   A5      690 non-null    str    
 5   A6      690 non-null    str    
 6   A7      690 non-null    str    
 7   A8      690 non-null    float64
 8   A9      690 non-null    str    
 9   A10     690 non-null    str    
 10  A11     690 non-null    int64  
 11  A12     690 non-null    str    
 12  A13     690 non-null    str    
 13  A14     690 non-null    str    
 14  A15     690 non-null    int64  
 15  Target  690 non-null    str    
dtypes: float64(2), int64(2), str(12)
memory usage: 86.4 KB


#### Convert strings to numeric

In [33]:
for c in df.columns:
    if df[c].dtype == "str":
        lbl = LabelEncoder()
        lbl.fit(list(df[c].values))
        df[c] = lbl.transform(list(df[c].values))

#### Separate features / target

In [34]:
X = df.drop(['Target'], axis=1)
y = df['Target']

#### Split train/test

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

#### Define search ranges for LogisticRegression and optimize

In [36]:
def objective_lgb(trial):
    penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
    solver = "liblinear" if penalty == "l1" else "lbfgs"

    params = {
        "C": trial.suggest_float("C", 1e-3, 100, log=True),
        "penalty": penalty,
        "solver": solver
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):

        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        pipeline = Pipeline(
            [
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(**params, max_iter=5000, random_state=42))
            ]
        )

        pipeline.fit(X_tr, y_tr)
        preds = pipeline.predict_proba(X_val)[:, 1]
        score = roc_auc_score(y_val, preds)
        scores.append(score)
        
    stdCV = np.std(scores)
    trial.set_user_attr("stdCV", stdCV)
    
    return np.mean(scores)


study_lgb = optuna.create_study(
    study_name="lgb", direction="maximize", sampler=optuna.samplers.TPESampler(seed=42)
)

study_lgb.optimize(objective_lgb, n_trials=10, show_progress_bar=True)
print("\nBest params:")
for k, v in study_lgb.best_trial.params.items():
    print(f"  {k}: {v}")

Best trial: 0. Best value: 0.932183: 100%|██████████| 10/10 [00:01<00:00,  7.78it/s]


Best params:
  penalty: l2
  C: 4.5705630998014515


#### Define search ranges for XGBClassifier and optimize

In [39]:
def objective_xgb(trial):

    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "n_estimators": 5000,
        "random_state": 42,
        "eval_metric": "auc",
        "early_stopping_rounds": 100,
        "tree_method": "hist"
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    best_iterations = []

    for train_idx, val_idx in cv.split(X_train, y_train):

        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = XGBClassifier(**params)

        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        preds = model.predict_proba(X_val)[:, 1]
        score = roc_auc_score(y_val, preds)
        scores.append(score)
        best_iterations.append(model.best_iteration)

    meanCV = np.mean(scores)
    stdCV = np.std(scores)
    median_best_iter = int(np.median(best_iterations))
    trial.set_user_attr("best_iteration", median_best_iter)
    trial.set_user_attr("stdCV", stdCV)

    return meanCV


study_xgb = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=42)
)
study_xgb.optimize(objective_xgb, n_trials=67, show_progress_bar=True)

print("\nBest params:")
for k, v in study_xgb.best_trial.params.items():
    print(f"  {k}: {v}")

Best trial: 65. Best value: 0.95444: 100%|██████████| 67/67 [01:25<00:00,  1.28s/it] 


Best params:
  max_depth: 6
  learning_rate: 0.05261339325050004
  subsample: 0.6209124934417534
  colsample_bytree: 0.9648700579845771
  reg_lambda: 4.282430788898925


#### Instantiate XGBClassifier with the best parameters

In [40]:
best_params = study_xgb.best_params
best_iteration = study_xgb.best_trial.user_attrs["best_iteration"]

final_model_xgb = XGBClassifier(
    **best_params,
    n_estimators=best_iteration,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    tree_method="hist",
    n_jobs=1
)

final_model_xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9648700579845771, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='auc', feature_types=None, feature_weights=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05261339325050004,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=79, n_jobs=1,
              num_parallel_tree=None, ...)

#### Instantiate the LogisticRegression Pipeline with the best parameters

In [41]:
best_params = study_lgb.best_params

final_model_lgb = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(**best_params, max_iter=5000, random_state=42))
    ]
)

final_model_lgb.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('clf',
                 LogisticRegression(C=4.5705630998014515, max_iter=5000,
                                    random_state=42))])

#### Function to calculate the confidence interval of the models

In [42]:
def bootstrap_auc(model, X, y, n_bootstrap=1000, random_state=42):
    rng = np.random.RandomState(random_state)
    aucs = []

    y_pred = model.predict_proba(X)[:, 1]

    for i in range(n_bootstrap):
        indices = rng.randint(0, len(y), len(y))
        if len(np.unique(y.iloc[indices])) < 2:
            continue 
        auc = roc_auc_score(y.iloc[indices], y_pred[indices])
        aucs.append(auc)

    return np.percentile(aucs, 2.5), np.percentile(aucs, 97.5), np.mean(aucs)

In [43]:
xgb_ci_low, xgb_ci_high, xgb_mean = bootstrap_auc(final_model_xgb, X_test, y_test)
lr_ci_low, lr_ci_high, lr_mean = bootstrap_auc(final_model_lgb, X_test, y_test)

#### Show metrics

In [44]:
df_study = pd.DataFrame(
    [
        {
            "mean CV": study_xgb.best_value,
            "std CV": study_xgb.best_trial.user_attrs["stdCV"],
            "AUC mean ": xgb_mean,
            "AUC 95% CI low": xgb_ci_low,
            "AUC 95% CI high": xgb_ci_high,
        },
        {
            "mean CV": study_lgb.best_value,
            "std CV": study_lgb.best_trial.user_attrs["stdCV"],
            "AUC mean ": lr_mean,
            "AUC 95% CI low": lr_ci_low,
            "AUC 95% CI high": lr_ci_high,
        },
    ]
)

df_study.rename(
    index={0: "XGBClassifier", 1: "LogisticRegression"},
    inplace=True
)
pd.set_option('display.precision', 4)
pd.set_option('display.float_format', '{:,.4f}'.format) 
df_study

,mean CV,std CV,AUC mean,AUC 95% CI low,AUC 95% CI high
XGBClassifier,0.9544,0.0169,0.9191,0.8666,0.9605
LogisticRegression,0.9322,0.0211,0.9005,0.8461,0.9504


#### Analysis of the results

**🔎 1. Mean CV**
- XGBClassifier: **0.9544**<br>
- LogisticRegression: 0.9322<br>
👉 XGBoost has better average performance.<br>
The difference is ~2.2 percentage points, which is significant.<br>

**📉 2. Stability (std CV)**
- XGBClassifier: **0.0169**<br>
- LogisticRegression: 0.0211<br>
👉 XGBoost not only performs better, but is **more stable** between folds (less variability).<br>

This indicates that:
- It generalizes better.<br>
- It is less sensitive to dataset partitioning.<br>

**🎯 3. AUC (discriminative capacity)**
- XGBClassifier: **0.9191**<br>
- LogisticRegression: 0.9005<br>
Both models have **very good discriminatory power** (>0.90 is excellent).<br>
However:<br>
- XGB has a better average AUC.<br>
- Its confidence interval is slightly larger.

**📏 4. Confidence intervals**
- XGB: [0.8666 – 0.9605]<br>
- LogReg: [0.8461 – 0.9504]<br>
The intervals overlap, suggesting that:<br>
- The difference may not be statistically significant.

**🧠 Technical Interpretation**

🔹 **XGBClassifier**<br>
✔ Better performance<br>
✔ Better stability<br>
✔ Better AUC<br>
❗ More complex model (lower interpretability)

🔹 **LogisticRegression**<br>
✔ Simpler<br>
✔ More interpretable<br>
✔ Lower risk of overfitting<br>
❗ Slightly lower performance

**🏁 Conclusion**

**XGBClassifier is superior in performance and stability**

#### XGBClassifier Optimization History 
![study_xgb](dataset/newplot.png "XGBClassifier Optimization History ")

In [ ]:
# from optuna.visualization import plot_optimization_history
# optuna.visualization.plot_optimization_history(study_xgb)

#### Predictions

In [45]:
y_pred = final_model_xgb.predict(X_test)
predictions = pd.DataFrame()
predictions['Actual'] = y_test
predictions['predicted'] = y_pred
predictions.sample(10)

,Actual,predicted
168,0,0
397,1,1
514,0,0
50,0,0
213,0,0
529,1,0
255,1,1
371,1,1
138,0,0
134,0,0


1 = Approval<br>
0 = Non-Approval